# DocVQA field labeling with GigaChat
Старая пилотная версия. Этот пайплайн не использовался в финальном эксперименте.

Ноутбук нужен как ранний черновой прогон разметки первых 200 примеров DocVQA по типу поля.

In [10]:
from collections import Counter
from pathlib import Path
import os
import re
import time

import pandas as pd
from datasets import load_dataset
from gigachat import GigaChat
from gigachat.models import Chat, Messages, MessagesRole
from tqdm.auto import tqdm

In [11]:
PROJECT_ROOT = Path.cwd().resolve().parent
ENV_PATH = PROJECT_ROOT / ".env"
OUTPUT_CSV = PROJECT_ROOT / "docvqa_field_types_pilot.csv"

DATASET_NAME = "pixparse/docvqa-single-page-questions"
SPLIT = "validation"
LIMIT = 200
MODEL = "GigaChat"
SLEEP_SEC = 0.2
VERIFY_SSL_CERTS = False

ALLOWED_TYPES = {
    "DATE",
    "AMOUNT",
    "NAME",
    "ID",
    "ADDRESS",
    "PHONE",
    "ORG",
    "PERCENTAGE",
    "REFERENCE",
    "OTHER",
}

PROMPT_TEMPLATE = """Ты помогаешь размечать датасет документов.

Тебе дан вопрос и ответ из документа. Определи тип поля документа, к которому относится ответ.

Вопрос: {question}
Ответ: {answer}

Выбери один тип из списка: DATE, AMOUNT, NAME, ID, ADDRESS, PHONE, ORG, PERCENTAGE, REFERENCE, OTHER

Если ни один не подходит — напиши новый тип в snake_case.
Отвечай ТОЛЬКО одним словом — названием типа. Никаких пояснений."""


def load_env_file(env_path: Path) -> None:
    if not env_path.exists():
        return

    for raw_line in env_path.read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


load_env_file(ENV_PATH)

GIGACHAT_CREDENTIALS = os.environ.get("GIGACHAT_CREDENTIALS", "").strip()
GIGACHAT_SCOPE = os.environ.get("GIGACHAT_SCOPE", "GIGACHAT_API_PERS").strip()

if not GIGACHAT_CREDENTIALS:
    raise RuntimeError("Environment variable GIGACHAT_CREDENTIALS is required.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {DATASET_NAME} [{SPLIT}] | limit={LIMIT}")
print(f"Output CSV: {OUTPUT_CSV}")

Project root: C:\Users\scfeel\DEV\hse\3year-2sem\course_work2026
Dataset: pixparse/docvqa-single-page-questions [validation] | limit=200
Output CSV: C:\Users\scfeel\DEV\hse\3year-2sem\course_work2026\docvqa_field_types_pilot.csv


In [12]:
dataset = load_dataset(DATASET_NAME, split=SPLIT)
pilot_dataset = dataset.select(range(min(LIMIT, len(dataset))))


def select_answer(example: dict) -> str:
    answers = example.get("answers") or []
    for answer in answers:
        text = str(answer).strip()
        if text:
            return text
    return ""


records = [
    {
        "question": str(example.get("question", "")).strip(),
        "answer": select_answer(example),
    }
    for example in pilot_dataset
]

pilot_df = pd.DataFrame(records)
pilot_df.head()

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

,question,answer
0,what is the page no or page mentioned ?,2
1,What is the name of the University?,Vanderbilt university
2,What is the Expenses for Salaries for 1986?,"$123,110"
3,What is the Expenses for Publications for 1987?,"10,596"
4,What is the Expenses for Publications for 1986?,"3,700"


In [13]:
client = GigaChat(
    credentials=GIGACHAT_CREDENTIALS,
    scope=GIGACHAT_SCOPE,
    model=MODEL,
    verify_ssl_certs=VERIFY_SSL_CERTS,
)


def normalize_field_type(raw_value: str) -> tuple[str, bool]:
    value = str(raw_value).strip()
    value = re.sub(r"[`\"'.,:;!?\s]+", "", value)
    value_upper = value.upper()

    if value_upper in ALLOWED_TYPES:
        return value_upper, False

    value_lower = value.lower()
    if re.fullmatch(r"[a-z][a-z0-9_]*", value_lower):
        return value_lower, True

    return "OTHER", False


def classify_field_type(question: str, answer: str) -> tuple[str, bool]:
    prompt = PROMPT_TEMPLATE.format(question=question, answer=answer)
    payload = Chat(
        messages=[Messages(role=MessagesRole.USER, content=prompt)],
        temperature=0,
    )
    response = client.chat(payload)
    raw_label = response.choices[0].message.content
    return normalize_field_type(raw_label)


In [14]:
results = []

for row in tqdm(records, total=len(records), desc="Labeling DocVQA"):
    question = row["question"]
    answer = row["answer"]

    try:
        field_type, is_new_type = classify_field_type(question, answer)
    except Exception:
        field_type, is_new_type = "ERROR", False

    results.append(
        {
            "question": question,
            "answer": answer,
            "field_type": field_type,
            "is_new_type": is_new_type,
        }
    )

    time.sleep(SLEEP_SEC)

results_df = pd.DataFrame(results)
results_df.head()

Labeling DocVQA:   0%|          | 0/200 [00:00<?, ?it/s]

,question,answer,field_type,is_new_type
0,what is the page no or page mentioned ?,2,REFERENCE,False
1,What is the name of the University?,Vanderbilt university,ORG,False
2,What is the Expenses for Salaries for 1986?,"$123,110",AMOUNT,False
3,What is the Expenses for Publications for 1987?,"10,596",AMOUNT,False
4,What is the Expenses for Publications for 1986?,"3,700",AMOUNT,False


In [15]:
results_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"Saved {len(results_df)} rows to: {OUTPUT_CSV}")

Saved 200 rows to: C:\Users\scfeel\DEV\hse\3year-2sem\course_work2026\docvqa_field_types_pilot.csv


In [16]:
field_stats = Counter(results_df["field_type"])
stats_df = pd.DataFrame(field_stats.items(), columns=["field_type", "count"]).sort_values(
    by="count",
    ascending=False,
    ignore_index=True,
)

print("Field type statistics:")
display(stats_df)

Field type statistics:


,field_type,count
0,AMOUNT,62
1,NAME,36
2,ORG,21
3,DATE,18
4,ID,16
5,person,9
6,REFERENCE,9
7,ADDRESS,7
8,PERCENTAGE,6
9,PHONE,6


In [18]:
other_df = results_df.loc[
    results_df["field_type"] == "OTHER",
    ["question", "answer", "field_type", "is_new_type"],
].reset_index(drop=True)

print(f"OTHER examples: {len(other_df)}")
display(other_df)

OTHER examples: 4


,question,answer,field_type,is_new_type
0,What should be done prior to B-3?,separate binder hydration in pulp,OTHER,False
1,What does EC stand for?,Error Correct,OTHER,False
2,What is the title in the last rectangle ?,available,OTHER,False
3,What is the second subheading?,Benefits,OTHER,False
